In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
from sklearn.feature_selection import SelectKBest, chi2

In [3]:
df_songs = pd.read_csv('dataset.csv')

In [14]:
def in_top(pred_list,actual, top_x):
    return True if actual in list(pred_list.keys())[:top_x] else False

def logistic_reg_and_score(df: pd.DataFrame, num_k: int):
    # Split the data into features (X) and target (y)
    X = df['lyrics_cleaned']
    y = df['parent_genre']

    # Split into training and test sets (80% training, 20% testing)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Create a pipeline: Vectorizer -> Logistic Regression
    # We use 'lbfgs' solver which is good for multiclass problems
    model_pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(stop_words='english',max_df=0.5, max_features=9000)),
        ('clf', LogisticRegression(solver='lbfgs', max_iter=1000,class_weight='balanced'))
    ])

    # Train the model
    print("Training model...")
    model_pipeline.fit(X_train, y_train)

    # Make predictions
    predictions = model_pipeline.predict(X_test)

    # Evaluate
    print(f"Accuracy: {accuracy_score(y_test, predictions):.2f}")
    print("\nClassification Report:\n")
    print(classification_report(y_test, predictions))

    # Get the probabilities
    probs = model_pipeline.predict_proba(X_test)

    # Get the genre names from the model
    genres = model_pipeline.classes_

    # Create a DataFrame where each column is a Genre
    y_prob = pd.DataFrame(probs, columns=genres, index=X_test.index)
    
    # This creates a dictionary of {Genre: Probability} for the top 3
    y_prob['predicted'] = y_prob.apply(
        lambda row: row.nlargest(3).to_dict(), 
        axis=1
    )

    df_pred = pd.DataFrame({
        'actual': y_test
    })

    df_pred = df_pred.join(y_prob['predicted'])
    for x in range(1,4):
        df_pred[f'top_{x}'] = df_pred.apply(lambda row: in_top(row['predicted'],row['actual'],x), axis=1)

    return df_pred

In [15]:
pred = logistic_reg_and_score(df_songs,10000)

Training model...
Accuracy: 0.39

Classification Report:

                  precision    recall  f1-score   support

       Asian Pop       0.43      0.43      0.43        96
       Classical       0.12      0.34      0.17        44
      Electronic       0.56      0.29      0.38       882
    Folk/Country       0.44      0.57      0.50       322
   Hip-Hop & R&B       0.22      0.59      0.31        29
    Jazz & Blues       0.13      0.32      0.18        57
           Latin       0.07      0.33      0.11        12
           Metal       0.53      0.71      0.61       452
             Pop       0.21      0.25      0.23       285
Reggae/Caribbean       0.46      0.53      0.49        86
            Rock       0.32      0.16      0.21       493
       Soul/Funk       0.18      0.28      0.22       157
  World/Regional       0.68      0.67      0.67       172

        accuracy                           0.39      3087
       macro avg       0.33      0.42      0.35      3087
    weighted

In [ ]:
num_words = []
top_1 = []
top_2 = []
top_3 = []

for x in range(7000,12500,500):
    pred = logistic_reg_and_score(df_songs,x)
    num_words.append(x)
    top_1.append(pred['top_1'].mean())
    top_2.append(pred['top_2'].mean())
    top_3.append(pred['top_3'].mean())

final_opt = pd.DataFrame({
    'num_words':num_words,
    'top_1_acc': top_1,
    'top_2_acc': top_2,
    'top_3_acc': top_3
})
